# 08b：已验证 Overlay 的板端回放与采集

此文件是接口入口，不包含可直接烧写的 P047Pro 固件。厂家 OFDM 的 DMA 只有 S2MM，不能用来发送任意 I/Q。
必须先完成 hardware/README.md 所述板级集成、时钟和采样映射验证，并在另一单元格得到已初始化的 `overlay`。
本 Notebook 默认不运行硬件，Python 不会远程连接板卡。


In [ ]:
from pathlib import Path
import sys, json
from datetime import datetime
from uuid import uuid4
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "afdm_lab").is_dir():
        ROOT = candidate.resolve()
        sys.path.insert(0, str(ROOT))
        break
else:
    raise RuntimeError("请先将整个 afdm_workbench.zip 解压；Notebook 不能脱离 afdm_lab 单独上传")
from afdm_lab.core import Config
from afdm_lab import workflows as work
cfg = Config()  # 默认 1 MHz 仅用于软件；不是自动测得的 RFDC 速率
out = ROOT / "afdm_runs" / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_08b_" + uuid4().hex[:6])
print("结果目录：", out)


In [ ]:
RUN_RF = False
PROFILE_PATH = ROOT / "configs" / "p047pro_unverified.json"
TX_FILE = None  # 07 生成的 tx.iq16 的完整路径
if RUN_RF:
    from afdm_lab.rf import capture_loaded_overlay
    profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
    # overlay 必须已经由用户按自己的板级工程配置完成。
    # 此函数不加载 bitstream、不设置 LMK/RFDC，也不改变通道接线。
    if TX_FILE is None:
        raise ValueError("先指定 07 的波形文件")
    rx_file = capture_loaded_overlay(overlay, TX_FILE, profile, out)
    print("实际采集文件：", rx_file)
    decoded = work.decode_file(rx_file, out.with_name(out.name + "_decode"))
    print(json.dumps(decoded, ensure_ascii=False, indent=2))
else:
    print("射频操作未执行：默认配置未验证，先完成板级回放/采集工程")
